<a href="https://colab.research.google.com/github/sergekamanzi/Fraudent-Docs/blob/main/drug.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [43]:
# Import Necessary Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model, save_model
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam, Adagrad
from tensorflow.keras.regularizers import l1
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.compose import ColumnTransformer
import joblib
from imblearn.over_sampling import SMOTE
from google.colab import files
import os
import joblib
from sklearn.preprocessing import StandardScaler, LabelEncoder  # Import necessary classes
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [44]:
data = pd.read_csv('/content/Drug prescription Dataset.csv')
data.head()

,disease,age,gender,severity,drug
0,diarrhea,4,male,LOW,promegranate drink
1,diarrhea,4,male,NORMAL,lime juice
2,diarrhea,5,male,LOW,promegranate drink
3,diarrhea,5,male,NORMAL,lime juice
4,diarrhea,6,male,LOW,promegranate drink


In [45]:
data.isna().sum()

,0
disease,0
age,0
gender,0
severity,0
drug,0


In [46]:
for column in data.columns:
    unique_values = data[column].unique()
    print(f"Unique values in '{column}': {unique_values}")

Unique values in 'disease': ['diarrhea' 'gastritis' 'arthritis' 'migraine']
Unique values in 'age': [ 4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24 25 26 27
 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48 49 50 51
 52 53 54 55 56 57 58 59 60]
Unique values in 'gender': ['male' 'female']
Unique values in 'severity': ['LOW' 'NORMAL' 'HIGH']
Unique values in 'drug': ['promegranate drink' 'lime juice' 'gandharvahastadi kashaya'
 'gulachyadi kashayam' 'laxative' 'nirugandi oil' 'nirugandi leaves paste'
 'ajwan water' 'ginger' 'fenugreek' 'shankhapushpi' 'haritaki' 'bilwa'
 'kutaja' 'hingwastaka churna' 'lavanaabhaskara churna' 'ajamodarka'
 'dashmool powder with water' 'dashmool oil' 'ashwagandha' 'trikatu'
 'shankh bhasma' 'dadimashtaka churna' 'kutajarishta' 'sanjni vati'
 'sankha vati' 'citrakadhi vati' 'hingvadi vati' 'shud laksha' 'ginger '
 'suhunjana beej' 'shankh-vati' 'drakshasava' 'pathyadi guggulu'
 'tribhuvan kirti rasa' 'mrutyunjay rasa' 'sanjiva

In [47]:
# Encode categorical variables
label_encoder = LabelEncoder()
data['disease'] = label_encoder.fit_transform(data['disease'])
data['gender'] = label_encoder.fit_transform(data['gender'])
data['severity'] = label_encoder.fit_transform(data['severity'])
data['drug'] = label_encoder.fit_transform(data['drug'])

In [48]:
data.head()

,disease,age,gender,severity,drug
0,1,4,1,1,50
1,1,4,1,2,41
2,1,5,1,1,50
3,1,5,1,2,41
4,1,6,1,1,50


In [49]:
# Split features and target
X = data.drop(columns=['drug'])
Y = data['drug']

In [52]:
# StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Save the scaler to a .pkl file
joblib.dump(scaler, 'scaler.pkl')

# Check the class distribution first
print("Class distribution before SMOTE:")
print(Y.value_counts())

# Apply SMOTE with adjusted k_neighbors
from imblearn.over_sampling import SMOTE

# Get the minimum number of samples in any class
min_samples = min(Y.value_counts())

# Set k_neighbors to be less than the minimum number of samples (but at least 1)
k_neighbors = min(5, max(1, min_samples - 1))

try:
    # Initialize SMOTE with adjusted k_neighbors
    smote = SMOTE(random_state=42, k_neighbors=k_neighbors)

    # Apply SMOTE to the scaled features and target
    X_resampled, y_resampled = smote.fit_resample(X_scaled, Y)

    print("Original dataset shape:", X_scaled.shape)
    print("Resampled dataset shape:", X_resampled.shape)
    print("Class distribution after SMOTE:")
    print(pd.Series(y_resampled).value_counts())

except ValueError as e:
    print(f"SMOTE failed: {e}")
    print("Consider removing classes with very few samples or using a different oversampling technique")

    # Alternative: Use SMOTE only if we have enough samples
    if min_samples > 2:
        smote = SMOTE(random_state=42, k_neighbors=min(2, min_samples - 1))
        X_resampled, y_resampled = smote.fit_resample(X_scaled, Y)
    else:
        print("Not enough samples for SMOTE. Using original data instead.")
        X_resampled, y_resampled = X_scaled, Y

# Download only the scaler file in Google Colab
files.download('scaler.pkl')

Class distribution before SMOTE:
drug
37    82
60    80
13    80
46    80
10    60
      ..
63     2
19     2
14     2
20     2
47     2
Name: count, Length: 65, dtype: int64
Original dataset shape: (1288, 4)
Resampled dataset shape: (5330, 4)
Class distribution after SMOTE:
drug
50    82
41    82
24    82
28    82
40    82
      ..
9     82
16    82
6     82
0     82
2     82
Name: count, Length: 65, dtype: int64


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>